In [ ]:
def bad_function(name):
    for i in range(10):
        print(name, i)
        sleep(0.1)


from dis import dis

In [ ]:
from greenlet_fiber import *

In [ ]:
from dis import Bytecode, Instruction, opmap
from types import FunctionType
import builtins


class Instructions(list):
    def __init__(s, f):
        super().__init__(Bytecode(f))
        s.F = f
        s.n = 0
        s.f = None
        s.name = f.__code__.co_names
        s.freevar = f.__code__.co_freevars
        s.const = f.__code__.co_consts
        s.closure = f.__closure__ or ()

    def seek(s, n) -> "Instructions":
        if s.f:
            s(None)
        s.n = n
        return s

    def __getattr__(s, f) -> "Instructions":
        if s.f:
            s(None)
        s.f = f
        return s

    def __call__(s, _, JUMP=("JUMP_IF_FALSE_OR_POP", "JUMP_IF_TRUE_OR_POP", "JUMP_ABSOLUTE", "POP_JUMP_IF_FALSE", "POP_JUMP_IF_TRUE", "JUMP_IF_NOT_EXC_MATCH"), SETUP=("SETUP_ANNOTATIONS", "SETUP_FINALLY", "SETUP_WITH", "SETUP_ASYNC_WITH", "FOR_ITER")) -> "Instructions":
        for j, i in enumerate(s):
            if (i.arg if i.opname in JUMP else j + i.arg + 1 if i.opname in SETUP and j < s.n else -1) >= s.n:
                s[j] = Instruction(*i[:2], i.arg + 1, *i[3:])
        if s.f == "LOAD_GLOBAL":
            i = "name"
            if not isinstance(_, str):
                s.F.__globals__[_.__name__] = _
                _ = _.__name__
        elif s.f == "LOAD_DEFER":
            i = "freevar"
            if not isinstance(_, str):
                if _ not in s.closure:
                    s.closure += (_,)
                    _ = _.__name__
                else:
                    _ = s.freevar[s.closure.index(_)]
        elif s.f == "LOAD_CONST":
            i = "const"
        else:
            i = None
        if i:
            j = getattr(s, i)
            if _ not in j:
                j += (_,)
                setattr(s, i, j)
            _ = j.index(_)
        s.insert(s.n, Instruction(s.f, opmap[s.f], _, None, None, None, None, None))
        s.f = None
        s.n += 1
        return s

    def build(s) -> FunctionType:
        if s.f:
            s(None)
        return FunctionType(s.F.__code__.replace(co_code=bytes(i for i in s for i in (i.opcode, i.arg or 0)), co_names=s.name, co_consts=s.const, co_freevars=s.freevar), s.F.__globals__, s.F.__name__, s.F.__defaults__, s.closure)


def inject(f, g, per=10) -> FunctionType:
    f = Instructions(f)
    for per in range(per, len(f), per)[::-1]:
        f.seek(per).LOAD_GLOBAL(g).CALL_FUNCTION.POP_TOP
    return f.build()


def wrap(f, g, depth=0) -> FunctionType:
    if hasattr(f, "__call__"):
        if hasattr(f, "__code__") and depth != 0:
            f = FunctionType(
                f.__code__.replace(
                    co_consts=tuple(wrap(i, g, depth - 1) or i for i in f.__code__.co_consts),
                    co_names=tuple([_ := "wrap" + i, f.__globals__.__setitem__(_, o)][0] if o else i for i in f.__code__.co_names for o in [wrap(f.__globals__.get(i, getattr(builtins, i, None)), g, depth - 1)]),
                ),
                f.__globals__,
                f.__name__,
                f.__defaults__,
                tuple(wrap(i, g, depth - 1) or i for i in f.__closure__ or ()),
            )
        return lambda *_, **__: g(f, *_, **__) or f(*_, **__)

In [ ]:
from random import choice
from signal import ITIMER_REAL, SIGALRM, setitimer, signal
from threading import current_thread
from greenlet import greenlet
from time import time

FIBERS = {}
TIMESLICE = 0.5


def get():
    return FIBERS.setdefault(current_thread(), ([], 0))


def fiber(f, *_, **__):
    @greenlet
    def fiber():
        try:
            f(*_, **__)
        finally:
            schedule(die=greenlet.getcurrent())

    get()[0].append(fiber)
    return fiber


def schedule(*_, die=None):
    fibers, last = get()
    if die:
        fibers.remove(die)
        for i in getattr(die, "rep", ()):
            i.dep -= 1
            if not i.dep:
                fibers.append(i)
    elif time() - last < TIMESLICE:
        return
    if fibers:
        cur = choice(fibers)
        if cur != greenlet.getcurrent():
            print(end=f"[schedule {current_thread().name}] {id(greenlet.getcurrent()):x} -> {id(cur):x}\n")
            FIBERS[current_thread()] = fibers, time()
            cur.switch()
    elif die:
        print(end=f"[schedule {current_thread().name}] ok\n")


def wait(*fibers, cur=None):
    cur = cur or greenlet.getcurrent()
    cur.dep = getattr(cur, "dep", 0) + len(fibers)
    for i in fiber:
        i.rep = getattr(i, "rep", [])
        i.rep.append(cur)
    try:
        get()[0].remove(cur)
    except ValueError:
        pass
    if cur == greenlet.getcurrent():
        schedule()


def waitmap(f, *_, **__):
    wait(*(fiber(f, *_) for _ in _), **__)


def setpreemptive(_=1):
    settimeslice(_)
    signal(SIGALRM, schedule)
    setitimer(ITIMER_REAL, _, _)


def settimeslice(_=1):
    global TIMESLICE
    TIMESLICE = _

In [ ]:
from time import sleep
from tqdm.notebook import tqdm


@lambda f: inject(f, schedule, 5)
@lambda f: [dis(f), f][1]
def f():
    for i in tqdm(range(100)):
        sleep(0.01)


fiber(f)
fiber(f)

schedule()

In [ ]:
from dis import dis

dis(f)